# Coleta `falando_nela` no Google Colab

Template para rodar os coletores em modo producao com Google Drive montado. O codigo fica no Git; os dados completos ficam no Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

os.environ['FALANDO_NELA_DATA_ROOT'] = '/content/drive/MyDrive/falando_nela/data'
Path(os.environ['FALANDO_NELA_DATA_ROOT']).mkdir(parents=True, exist_ok=True)
print(os.environ['FALANDO_NELA_DATA_ROOT'])

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


## Parametros

Use uma janela curta primeiro para validar escrita no Drive. Depois rode o baseline completo.

In [ ]:
DATA_INICIO = '2011-05-18'
DATA_FIM = '2026-05-18'

# Para validacao inicial no Drive, descomente uma janela curta:
# DATA_INICIO = '2026-02-01'
# DATA_FIM = '2026-02-28'

COLETORES = [
    'coleta.senado.plenario_discursos.collect',
    'coleta.senado.congresso_discursos.collect',
    'coleta.senado.ccj_notas.collect',
    'coleta.senado.pareceres_pec.collect',
    'coleta.camara.plenario_discursos.collect',
    'coleta.camara.ccjc_eventos.collect',
    'coleta.camara.pareceres_pec.collect',
]

In [ ]:
import subprocess
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
manifest_paths = []

for module in COLETORES:
    dataset = module.replace('coleta.', '').replace('.collect', '').replace('.', '-')
    run_id = f'prod-{dataset}-{run_stamp}'
    cmd = [
        'python', '-m', module,
        '--mode', 'prod',
        '--resume',
        '--run-id', run_id,
        '--data-inicio', DATA_INICIO,
        '--data-fim', DATA_FIM,
    ]
    print('Rodando:', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, text=True, capture_output=True)
    print(result.stdout)
    manifest_paths.append(result.stdout.strip().splitlines()[-1])

manifest_paths

In [ ]:
import json
from pathlib import Path

for manifest_path in manifest_paths:
    path = Path(manifest_path)
    manifest = json.loads(path.read_text(encoding='utf-8'))
    print('\nMANIFEST:', path)
    print(json.dumps({
        'source': manifest.get('source'),
        'dataset': manifest.get('dataset'),
        'mode': manifest.get('mode'),
        'sample': manifest.get('sample'),
        'record_counts': manifest.get('record_counts'),
        'partition_counts': manifest.get('partition_counts'),
    }, ensure_ascii=False, indent=2))
    log_path = Path(manifest['log_path'])
    if log_path.exists():
        print('Ultimas linhas do log:')
        print('\n'.join(log_path.read_text(encoding='utf-8').splitlines()[-5:]))